# 01. Graph-Based RAG (그래프 기반 RAG)

> 📌 **용어 주의**: 여기서 "Graph-Based"는 **LangGraph `StateGraph`로 RAG 파이프라인을 조립한다**는 뜻이에요. **지식 그래프(Knowledge Graph)와 Cypher 질의**로 관계형 질문을 처리하는 쪽은 **`11_Use_Cases/06-GraphRAG-Neo4j.ipynb`** 에 따로 있어요. 이 챕터(08)의 1~5 레슨은 모두 **벡터 기반 RAG**이고, 지식 그래프 기반은 11장의 use case로 연결돼요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. RAG(Retrieval-Augmented Generation)의 3가지 종류와 각 특징을 설명할 수 있어요
2. `StateGraph`로 Retrieve → Generate 2단계 RAG 파이프라인을 직접 구성할 수 있어요
3. `MemorySaver` 체크포인터와 `thread_id`로 멀티턴 대화 상태를 유지할 수 있어요
4. `add_conditional_edges`로 관련성 판단 기반 재검색 루프를 만들 수 있어요

## 사전 지식

- 이전 파트(Part 07)에서 배운 메모리 체크포인터(`MemorySaver`)와 `thread_id`
- `StateGraph`, `State`, `Node`, `Edge` 기초 (Part 02 참고)
- `MemorySaver`와 `thread_id`를 이용한 체크포인팅 (Part 02-06 참고)

## RAG란 무엇인가요?

**RAG(Retrieval-Augmented Generation)**는 LLM이 답변을 생성할 때 외부 지식 베이스에서 관련 문서를 먼저 검색하여 활용하는 기법이에요.
LLM의 학습 데이터 제한과 환각(Hallucination) 문제를 해결하는 핵심 패턴이에요.

> 🔑 **핵심 개념**: RAG는 **도서관 사서**와 같아요. 질문을 받으면 (1) 서가에서 관련 책을 찾고(Retrieve), (2) 그 책을 참고해서 답변을 작성해요(Generate). 사서가 책 없이 기억에만 의존하면 잘못된 정보를 줄 수 있듯이, LLM도 검색 없이 답변하면 환각(Hallucination)이 발생해요.

### 왜 RAG가 필요한가요?

LLM만으로 답변할 때 발생하는 문제를 정리해볼게요:

| 문제 | 증상 | RAG의 해결 방법 |
|------|------|----------------|
| **학습 데이터 한계** | 2024년 이후 정보를 모름 | 최신 문서를 검색해서 참고 |
| **환각(Hallucination)** | 그럴듯하지만 틀린 답변 | 실제 문서 근거로 답변 |
| **도메인 지식 부재** | 사내 문서 내용을 모름 | 사내 문서를 벡터화해서 검색 |
| **출처 불명** | "어디서 봤는데..." | 문서명 + 페이지 번호 명시 |

### RAG의 3가지 종류

| 종류 | 흐름 | 특징 | 적합한 경우 |
|------|------|------|------------|
| **Naive RAG** (2-Step) | 검색 → 생성 | 단순, 빠름 | 단순한 Q&A |
| **Agentic RAG** | 검색 → 판단 → 재검색 | 자율적 판단 | 복잡한 질문 |
| **Hybrid RAG** | 벡터 + 키워드 검색 | 높은 재현율 | 대규모 문서 |

이 노트북에서는 **Naive RAG(2-Step RAG)**와 **재검색 루프**를 LangGraph로 구현해요.

> 🔑 **핵심 개념**: RAG 파이프라인 5단계 - **Load(로드) → Split(분할) → Embed(임베딩) → Store(저장) → Retrieve+Generate(검색+생성)**. LangGraph는 마지막 Retrieve+Generate 단계를 노드로 분리하여 제어 흐름을 명시적으로 만들어줘요.

### 전체 아키텍처

```mermaid
flowchart TD
    User([사용자 질문<br/>User Question])
    Retrieve[retrieve<br/>문서 검색]
    LLMAnswer[llm_answer<br/>답변 생성]
    Relevance[relevance_check<br/>관련성 판단]
    Result([최종 답변<br/>Final Answer])

    User --> Retrieve
    Retrieve --> LLMAnswer
    LLMAnswer --> Result

    Retrieve2[retrieve<br/>재검색]
    Relevance2[relevance_check<br/>관련성 판단]

    User2([사용자 질문<br/>User Question]) --> Retrieve2
    Retrieve2 --> LLMAnswer2[llm_answer<br/>답변 생성]
    LLMAnswer2 --> Relevance2
    Relevance2 -- 관련성 낮음 --> Retrieve2
    Relevance2 -- 관련성 높음 --> Result2([최종 답변])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef decision fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class User,User2 input
    class Retrieve,LLMAnswer,Retrieve2,LLMAnswer2 process
    class Relevance,Relevance2 decision
    class Result,Result2 output
```

> 💡 **핵심 정리**: 왼쪽(Naive RAG)과 오른쪽(재검색 루프)의 차이를 확인해요. 재검색 루프는 `add_conditional_edges`로 구현되며, 관련성이 낮으면 다시 검색을 수행해요. 이것이 "Agentic" RAG의 핵심이에요.

## 환경 설정

In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키를 가져와요)
from dotenv import load_dotenv

load_dotenv(override=True)
# 환경 변수 로드 완료

In [ ]:
# LangSmith 추적 설정 (RAG 파이프라인 실행 흐름을 시각화할 수 있어요)
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-08-RAG"

# LangSmith 추적 설정 완료

## 1. PDF 문서 기반 Retriever 구성

RAG의 첫 단계는 문서를 벡터 저장소에 인덱싱하는 거예요.
PDF를 로드하고, 청크(Chunk)로 분할한 뒤, 임베딩 벡터로 변환하여 FAISS 저장소에 저장해요.

### PDF 파이프라인 구성

```mermaid
flowchart LR
    PDF([PDF 문서]) --> Loader[PyPDFLoader<br/>PDF 로드]
    Loader --> Splitter[RecursiveCharacterTextSplitter<br/>청크 분할]
    Splitter --> Embedder[OpenAIEmbeddings<br/>벡터 변환]
    Embedder --> Store[(FAISS<br/>벡터 저장소)]
    Store --> Retriever[Retriever<br/>유사도 검색]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class PDF input
    class Loader,Splitter,Embedder process
    class Store storage
    class Retriever output
```

> 🔑 **핵심 개념**: 벡터 검색은 **의미 기반 검색 엔진**이에요. 키워드가 정확히 일치하지 않아도 "투자금액"과 "자금 투입"처럼 의미가 비슷한 내용을 찾아줘요. 문장을 숫자 벡터(좌표)로 변환한 뒤, 가까운 좌표에 있는 문서를 "관련 문서"로 판단해요.

> 💡 **실무 팁**: `chunk_size=1000, chunk_overlap=200`은 검색 품질과 토큰 비용의 균형을 맞춘 설정이에요. 청크가 너무 작으면 맥락이 끊기고, 너무 크면 LLM 컨텍스트 윈도우를 낭비해요.

> ⚠️ **자주 하는 실수**: 임베딩 모델과 벡터 저장소를 매번 새로 만들면 비용이 많이 들어요. 실무에서는 `CacheBackedEmbeddings`로 임베딩 결과를 캐시하거나, 저장소를 디스크에 저장해두고 재사용해요.

**실습용 문서**: 소프트웨어정책연구소(SPRi) AI Brief 2023년 12월호 PDF
- `data/SPRI_AI_Brief_2023년12월호_F.pdf`

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import os

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### Retriever 테스트

실제로 질문을 넣어서 검색이 잘 되는지 확인해볼게요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 검색 테스트: 질문과 유사한 문서를 찾아요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. RAG 체인 구성

검색된 문서를 LLM 프롬프트에 담아 답변을 생성하는 체인을 만들어요.

> 🔑 **핵심 개념**: `format_docs` 함수는 `Document` 객체 리스트를 LLM이 읽을 수 있는 텍스트로 변환해요. XML 태그로 각 문서를 감싸면 LLM이 문서 경계를 명확히 이해해요 — **간접 프롬프트 인젝션 방어** 효과도 있어요.

> 💡 **실무 팁**: 컨텍스트를 `<document>` 태그로 묶으면 두 가지 이점이 있어요. 첫째, LLM이 "이 내용이 사용자 입력이 아닌 문서"임을 인지해요. 둘째, 프롬프트 인젝션 공격(악성 문서에 포함된 명령)을 방어해요.

In [ ]:
from langchain_core.documents import Document
from typing import List

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 테스트: 검색 결과를 포맷팅해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 체인 단독 테스트 (그래프 없이 체인만 실행해볼게요)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. State 정의

`State`는 그래프의 모든 노드가 공유하는 데이터 구조예요.
RAG 파이프라인에서는 질문, 검색된 문서, 생성된 답변, 대화 기록이 필요해요.

### State 설계 원칙

| 필드 | 타입 | Reducer | 설명 |
|------|------|---------|------|
| `question` | `str` | 덮어쓰기 | 현재 사용자 질문 |
| `context` | `str` | 덮어쓰기 | 검색된 문서 (포맷팅 후) |
| `answer` | `str` | 덮어쓰기 | 생성된 최종 답변 |
| `messages` | `list` | `add_messages` | 대화 히스토리 (누적) |

> 🔑 **핵심 개념**: `add_messages` 리듀서는 리스트를 **덮어쓰지 않고 누적**해요. 여러 노드에서 `messages`를 반환해도 모두 합쳐져요. 반면 `question`, `context`, `answer`는 마지막 값으로 덮어써요.

> 💡 **실무 팁**: State에는 **raw 데이터**를 저장하세요. 포맷팅(format_docs)은 노드 안에서 처리하는 것이 더 유연해요. State를 직렬화하기 쉽고, 여러 노드에서 다른 방식으로 포맷팅할 수 있어요.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 노드(Node) 정의

RAG 파이프라인을 **검색 노드**와 **생성 노드**로 분리해요.
분리하면 각 노드를 독립적으로 테스트하고 교체할 수 있어요.

### 노드 책임 분리

```mermaid
flowchart LR
    Input([question]) --> Retrieve[retrieve_document<br>❶ 검색 담당]
    Retrieve --> ContextUpdate([context 업데이트])
    ContextUpdate --> LLMAnswer[llm_answer<br>❷ 생성 담당]
    LLMAnswer --> OutputUpdate([answer + messages 업데이트])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class Input,ContextUpdate,OutputUpdate input
    class Retrieve,LLMAnswer process
```

> 💡 **핵심 정리**: 노드를 분리하는 이유를 살펴봅니다. 검색 노드를 웹 검색으로 교체하거나, 생성 노드의 모델을 변경할 때 다른 노드에 영향이 없어요. 이것이 "모듈식 설계"의 장점이에요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. Naive RAG 그래프 구성

가장 단순한 RAG 파이프라인이에요: **START → retrieve → llm_answer → END**

MemorySaver 체크포인터를 추가하면 멀티턴 대화를 지원해요.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → retrieve → llm_answer → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. 그래프 실행

**단일 질문 실행**부터 시작해서 **멀티턴 대화**까지 테스트해볼게요.

> ⚠️ **자주 하는 실수**: 같은 `thread_id`로 다시 실행하면 이전 대화가 이어져요. 새로운 대화를 시작하려면 새로운 `thread_id`를 사용하세요.

In [ ]:
import uuid
from langchain_core.runnables import RunnableConfig

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. 관련성 판단 + 재검색 루프

검색된 문서가 질문과 무관하다면 다시 검색을 시도하는 루프를 만들어볼게요.
`add_conditional_edges`로 분기점을 추가하면 돼요.

### 재검색 루프 아키텍처

```mermaid
flowchart TD
    START([START]) --> Retrieve[retrieve<br>문서 검색]
    Retrieve --> LLMAnswer[llm_answer<br>답변 생성]
    LLMAnswer --> RelevanceCheck[relevance_check<br>관련성 판단]
    RelevanceCheck -- yes: 관련 있음 --> END([END])
    RelevanceCheck -- no: 관련 없음 --> Retrieve

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef decision fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef terminal fill:#d4edda,stroke:#28a745,color:#155724

    class Retrieve,LLMAnswer process
    class RelevanceCheck decision
    class START,END terminal
```

> 🔑 **핵심 개념**: `add_conditional_edges`의 마지막 인자는 **라우팅 맵**이에요. 라우팅 함수가 반환하는 문자열 키를 실제 노드 이름(또는 END)으로 매핑해요.

> ⚠️ **자주 하는 실수**: `recursion_limit`을 너무 작게 설정하면 재검색 도중에 그래프가 종료돼요. 재검색이 3번 이상 일어날 수 있는 경우 `recursion_limit=20` 이상으로 설정하세요.

In [ ]:
from langchain_core.prompts import PromptTemplate

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → retrieve → llm_answer → relevance_check → (END | retrieve)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. 실습: 나만의 RAG 파이프라인 커스터마이징

아래 TODO 블록을 수정해서 다양한 설정을 실험해봐요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 아래 파라미터를 변경해서 RAG 성능 차이를 실험해보세요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 9. 주요 패턴 비교

이 노트북에서 구현한 두 가지 RAG 패턴을 비교해볼게요.

| 패턴 | 그래프 구조 | 장점 | 단점 |
|------|-----------|------|------|
| **Naive RAG** | retrieve → llm_answer | 단순, 빠름 | 관련성 미검증 |
| **재검색 루프** | retrieve → llm_answer → relevance_check → (루프) | 품질 보장 | LLM 호출 추가 |

> 💡 **핵심 정리**: 재검색 루프는 관련성 판단에 LLM을 한 번 더 호출해요. "비용 vs 품질" 트레이드오프를 살펴봅니다. 실무에서는 빠른 임베딩 기반 코사인 유사도 점수가 0.7 미만일 때만 재검색하는 방식을 많이 사용해요.

> 💡 **실무 팁**: LangSmith 추적을 활성화했다면 각 노드의 실행 시간과 LLM 토큰 사용량을 확인할 수 있어요. 재검색 루프가 평균 몇 번 발생하는지 분석하면 파이프라인을 최적화할 수 있어요.

## RAG 시스템의 흔한 함정들

RAG 시스템을 처음 구축할 때 자주 빠지는 함정들을 정리했어요.

> ⚠️ **자주 하는 실수**: 아래 5가지는 RAG 프로젝트에서 가장 빈번한 실수예요.

| 함정 | 증상 | 해결 방법 |
|------|------|----------|
| **청크가 너무 큼** | 답변에 관련 없는 내용이 많음 | `chunk_size`를 500~1000자로 줄이세요 |
| **청크가 너무 작음** | 핵심 문맥이 잘림 | `chunk_overlap`을 200자 이상으로 늘리세요 |
| **검색 결과 미검증** | 환각(Hallucination) 발생 | Groundedness Check 노드를 추가하세요 |
| **임베딩 캐시 미사용** | API 비용 급증 | `CacheBackedEmbeddings`로 캐시하세요 |
| **프롬프트에 출처 미포함** | 답변 신뢰도 저하 | XML 태그로 출처를 프롬프트에 포함하세요 |

> 💡 **실무 팁**: 첫 번째 RAG 시스템은 Naive RAG로 만들고, LangSmith로 검색 품질을 모니터링하면서 점진적으로 개선하는 것이 가장 효과적이에요. 처음부터 Adaptive RAG를 적용하면 디버깅이 어려워져요.

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Naive RAG**: `retrieve → llm_answer` 2단계 파이프라인. 단순하지만 관련성 미검증이라는 한계가 있어요
- **format_docs**: `Document` 리스트를 XML 태그로 변환하여 LLM이 문서 경계를 명확히 인식하게 해요. 간접 프롬프트 인젝션도 방어해요
- **GraphState**: `question`, `context`, `answer`, `messages` 필드로 RAG 파이프라인 상태를 관리해요. `messages`는 `add_messages` 리듀서로 누적돼요
- **MemorySaver + thread_id**: 대화 세션별로 상태를 저장하여 멀티턴 대화를 지원해요
- **add_conditional_edges**: 관련성 판단 결과에 따라 재검색 또는 종료를 결정하는 루프를 만들 수 있어요
- **노드 분리**: 검색(retrieve)과 생성(llm_answer)을 독립 노드로 분리하면 각각 교체·테스트가 쉬워요

## 다음 노트북 예고

### Part 08 전체 로드맵

이 노트북에서 **Naive RAG**의 기본기를 다졌어요. Part 08에서는 점진적으로 더 강력한 RAG 시스템을 만들어가요:

| 노트북 | 핵심 질문 | 추가되는 기능 |
|--------|----------|-------------|
| **01 (현재)** | "문서에서 찾아줘" | Retrieve → Generate 기본 파이프라인 |
| **02 Agentic** | "검색할지 말지 알아서 판단해" | 에이전트 자율 도구 선택 + Groundedness |
| **03 CRAG/Self** | "답변이 정말 맞는지 검증해" | 환각 검증 + 자기 교정 루프 |
| **04 Adaptive** | "어디서 검색할지 먼저 결정해" | 질문 라우팅 + 3단계 검증 |
| **05 Retrieval** | "검색 자체를 더 똑똑하게" | 쿼리 강화 + MMR + 메타데이터 필터링 |

다음 `02-Agentic-RAG.ipynb`에서는 **에이전트 기반 RAG(Agentic RAG)**를 배워요. 단순한 검색 루프를 넘어, 에이전트가 자율적으로 검색 전략을 결정하고 Groundedness Check(근거 기반 판단)와 웹 검색을 결합하는 더 강력한 RAG 시스템을 구현해요.